# Direct Orthogonal Feature Correction 


In [1]:
# Environment and reproducibility setup
# Run this once if the imports below are missing:
# %pip install "numpy>=1.26" "pandas>=2.1" "matplotlib>=3.8" \
#     "pillow>=10" "scikit-learn>=1.4" "torch>=2.2" \
#     "transformers>=4.56" "accelerate>=0.30"

from __future__ import annotations

import json
import random
import sys
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def package_versions(names=("numpy", "pandas", "scikit-learn", "torch", "transformers")):
    rows = []
    for name in names:
        try:
            installed = version(name)
        except PackageNotFoundError:
            installed = "not installed"
        rows.append({"package": name, "version": installed})
    return pd.DataFrame(rows)

print(f"Python {sys.version.split()[0]}")
package_versions()


Python 3.10.9


,package,version
0,numpy,1.26.4
1,pandas,2.2.3
2,scikit-learn,1.6.1
3,torch,1.11.0
4,transformers,not installed


This approach operates on the internal mathematics of the drone's vision system, aiming to align how a frozen vision foundation model (VFM) represents the world across different flight heights


In [2]:
# Project-wide configuration. Change only these values when the aircraft changes.
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PATCH_DIR = DATA_DIR / "patches_fixed_canvas"
FEATURE_DIR = DATA_DIR / "features"
RESULT_DIR = REPO_ROOT / "results" / "q3a"
MANIFEST_PATH = DATA_DIR / "manifest.csv"

# Tello pilot. Change to 30.0 and 80.0 when using a suitable aircraft.
LOW_ALTITUDE_M = 10.0
HIGH_ALTITUDE_M = 25.0
PATCH_SIZE_PX = (256, 256)  # width, height; fixed at both altitudes
MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"
FALLBACK_MODEL_ID = "facebook/dinov2-small"

for directory in (RAW_DIR, PATCH_DIR, FEATURE_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "low_altitude_m": LOW_ALTITUDE_M,
    "high_altitude_m": HIGH_ALTITUDE_M,
    "patch_size_px": PATCH_SIZE_PX,
    "model_id": MODEL_ID,
    "seed": SEED,
}
print(json.dumps(CONFIG, indent=2))


{
  "low_altitude_m": 10.0,
  "high_altitude_m": 25.0,
  "patch_size_px": [
    256,
    256
  ],
  "model_id": "facebook/dinov3-vits16-pretrain-lvd1689m",
  "seed": 42
}


## Cross-altitude feature correction

### The Three Core Operational Pillars
To rigorously evaluate whether altitude shifts operate as a structured, reversible transformation, the experiment is defined by three strict boundaries:
1. Frozen Encoder: The parameters and feature-layer of the vision foundation model remain completely locked. The model is strictly used as a static feature extractor
2. Fixed 30m Probe: A linear classifier is trained only on real 30m features. Its weights, intercept, and decision boundaries are frozen and never updated during high-altitude testing
3. Target-Calibrated Correction: An alignment matrix is computed using a separate, matched set of real 30m and 80m feature pairs representing the same physical ground objects (cars, road markings, etc.)

### The Mathematics of Orthogonal Procrustes
The correction relies on Orthogonal Procrustes Analysis, which seeks the optimal high-dimensional rotation and reflection matrix to map high-altitude features back into the low-altitude baseline space.

**A. Strict Orthogonal Mapping (Primary)**
The closed-form solution is computed using Singular Value Decomposition (SVD)
1. Calculate the cross-covariance matrix: $X_{80}^T X_{30}$
2. Compute its SVD: $X_{80}^T X_{30} = U \Sigma V^T$
3. Construct the optimal rotation matrix: $R^* = U V^T$
4. Apply the strict mapping to correct test features: $\hat{z}_{30} = z_{80} R^*$

**B. Centred Orthogonal Mapping (Secondary)**
A common variation centers each feature domain around its mean vector before applying the rotation, adding a translation (mean shift) to the correction
$$\hat{z}_{30} = (z_{80} - \mu_{80}) R^* + \mu_{30}$$

### Isolating the Transform: The 7 Study Arms
To scientifically prove that the correction works, the performance of the unchanged 30m linear probe must be compared across seven distinct test conditions evaluated on the same held-out test objects

### Critical Methodological Safeguards
To ensure the academic validity of the study, three strict safeguards must be implemented:
- Grouped Data Splitting (Preventing Leakage): Adjacent video frames, multiple crops, and all altitudes of a single physical object must belong to the exact same partition. Must enforce a strict group-based split (such as GroupKFold) based on physical object identity. If a physical object is used to estimate the Procrustes mapping (Calibration), it must never appear in the Training or Final-Test partitions
- The Scale-Preserving Patch Protocol: When cropping targets, must extract a fixed native-pixel canvas size around the object center at both altitudes rather than cropping tightly and resizing. Resizing every crop to a uniform network size removes the very visual scale/GSD variable
- Rank and Dimensionality Limits: Because modern foundation model embeddings are very high-dimensional (typically 768 or 1024) and the calibration set of independent physical objects might be smaller, the centered covariance matrix has a rank of at most $n - 1$. The study handles this by comparing the full representation with a lower-dimensional projection (like PCA) fitted only on calibration or development data

Ultimately, the strict orthogonal correction (Arm C) is considered successful only if it improves the chosen task metric (such as Macro-F1 or balanced accuracy) on held-out real 80m objects by a practically useful amount, and the underlying feature alignment diagnostics move in the expected direction


In [3]:
# Dataset manifest, leakage checks, object-level splitting, and fixed-canvas crops
MANIFEST_COLUMNS = [
    "sample_id", "object_id", "class_name", "altitude_m", "image_path",
    "center_x", "center_y", "session_id", "site_id", "split",
]

def empty_manifest() -> pd.DataFrame:
    return pd.DataFrame(columns=MANIFEST_COLUMNS)

if not MANIFEST_PATH.exists():
    empty_manifest().to_csv(MANIFEST_PATH, index=False)
    print(f"Created manifest template: {MANIFEST_PATH}")
else:
    print(f"Using existing manifest: {MANIFEST_PATH}")

def resolve_repo_path(value: str | Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else REPO_ROOT / path

def validate_manifest(df: pd.DataFrame, check_files: bool = False) -> None:
    missing = sorted(set(MANIFEST_COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Missing manifest columns: {missing}")
    if df["sample_id"].duplicated().any():
        duplicates = df.loc[df["sample_id"].duplicated(), "sample_id"].tolist()
        raise ValueError(f"Duplicate sample_id values: {duplicates[:5]}")
    inconsistent = df.groupby("object_id")["class_name"].nunique()
    if (inconsistent > 1).any():
        raise ValueError("Each physical object_id must have exactly one class_name")
    if df["split"].notna().any():
        leakage = df.groupby("object_id")["split"].nunique()
        if (leakage > 1).any():
            raise ValueError("Object leakage detected across data partitions")
    required_altitudes = {LOW_ALTITUDE_M, HIGH_ALTITUDE_M}
    observed = df.groupby("object_id")["altitude_m"].apply(lambda x: set(map(float, x)))
    incomplete = observed[~observed.apply(required_altitudes.issubset)]
    if len(incomplete):
        print(f"Warning: {len(incomplete)} objects do not yet contain both study altitudes")
    if check_files:
        missing_files = [p for p in df["image_path"] if not resolve_repo_path(p).exists()]
        if missing_files:
            raise FileNotFoundError(f"Missing image files, first examples: {missing_files[:5]}")

def _stratify_if_possible(labels: pd.Series):
    counts = labels.value_counts()
    return labels if len(counts) > 1 and counts.min() >= 2 else None

def assign_object_splits(
    df: pd.DataFrame,
    fractions=None,
    random_state: int = SEED,
) -> pd.DataFrame:
    fractions = fractions or {
        "source_train": 0.40,
        "calibration": 0.25,
        "development": 0.15,
        "final_test": 0.20,
    }
    if not np.isclose(sum(fractions.values()), 1.0):
        raise ValueError("Split fractions must sum to 1")

    objects = df.groupby("object_id", as_index=False).agg(class_name=("class_name", "first"))
    remaining, final_test = train_test_split(
        objects,
        test_size=fractions["final_test"],
        random_state=random_state,
        stratify=_stratify_if_possible(objects["class_name"]),
    )
    remaining_fraction = 1.0 - fractions["final_test"]
    development_share = fractions["development"] / remaining_fraction
    remaining, development = train_test_split(
        remaining,
        test_size=development_share,
        random_state=random_state + 1,
        stratify=_stratify_if_possible(remaining["class_name"]),
    )
    calibration_share = fractions["calibration"] / (
        fractions["source_train"] + fractions["calibration"]
    )
    source_train, calibration = train_test_split(
        remaining,
        test_size=calibration_share,
        random_state=random_state + 2,
        stratify=_stratify_if_possible(remaining["class_name"]),
    )

    mapping = {}
    for name, subset in {
        "source_train": source_train,
        "calibration": calibration,
        "development": development,
        "final_test": final_test,
    }.items():
        mapping.update({object_id: name for object_id in subset["object_id"]})

    result = df.copy()
    result["split"] = result["object_id"].map(mapping)
    validate_manifest(result)
    return result

def crop_fixed_canvas(
    image: Image.Image,
    center_xy: tuple[float, float],
    size_px: tuple[int, int] = PATCH_SIZE_PX,
    fill=(0, 0, 0),
) -> Image.Image:
    image = image.convert("RGB")
    width, height = map(int, size_px)
    cx, cy = map(float, center_xy)
    left = int(round(cx - width / 2))
    top = int(round(cy - height / 2))
    right, bottom = left + width, top + height

    canvas = Image.new("RGB", (width, height), fill)
    src_box = (
        max(left, 0), max(top, 0),
        min(right, image.width), min(bottom, image.height),
    )
    if src_box[2] <= src_box[0] or src_box[3] <= src_box[1]:
        raise ValueError("Patch centre lies outside the image")
    region = image.crop(src_box)
    canvas.paste(region, (src_box[0] - left, src_box[1] - top))
    return canvas

def materialize_fixed_canvas_patches(df: pd.DataFrame) -> pd.DataFrame:
    validate_manifest(df, check_files=True)
    result = df.copy()
    patch_paths = []
    for row in result.itertuples(index=False):
        source_path = resolve_repo_path(row.image_path)
        output_path = PATCH_DIR / f"{row.sample_id}.png"
        with Image.open(source_path) as image:
            patch = crop_fixed_canvas(
                image,
                center_xy=(row.center_x, row.center_y),
                size_px=PATCH_SIZE_PX,
            )
            patch.save(output_path)
        patch_paths.append(output_path.relative_to(REPO_ROOT).as_posix())
    result["patch_path"] = patch_paths
    return result

empty_manifest().head()


Created manifest template: C:\Users\ADMIN\Desktop\fastr\uav-altitude-feature-alignment\data\manifest.csv


,sample_id,object_id,class_name,altitude_m,image_path,center_x,center_y,session_id,site_id,split


In [4]:
# Optional dimensionality reduction for n_calibration << feature_dimension.
# Fit only on permitted source/development data, never on final-test objects.
def fit_source_pca(X_source: np.ndarray, n_components: int) -> PCA:
    maximum = min(X_source.shape[0] - 1, X_source.shape[1])
    if not 1 <= n_components <= maximum:
        raise ValueError(f"n_components must be between 1 and {maximum}")
    return PCA(n_components=n_components, whiten=False, random_state=SEED).fit(X_source)


## Orthogonal Procrustes

Orthogonal Procrustes mapping acts as a high-dimensional spatial alignment tool. It seeks to find a single optimal rotation and reflection matrix that mathematically rotates the 80m feature cluster so that it aligns directly with the 30m cluster. This allows a simple classifier (a linear probe) trained exclusively on 30m features to work seamlessly on 80m data


In [5]:
# A self-contained recovery test: known orthogonal geometry should be recoverable.
def solve_orthogonal_procrustes(X_source: np.ndarray, X_target: np.ndarray):
    X_source = np.asarray(X_source, dtype=np.float64)
    X_target = np.asarray(X_target, dtype=np.float64)
    if X_source.shape != X_target.shape or X_source.ndim != 2:
        raise ValueError("Source and target must be 2-D matrices with identical shapes")
    U, singular_values, Vt = np.linalg.svd(X_source.T @ X_target, full_matrices=False)
    R = U @ Vt
    return R, singular_values

rng = np.random.default_rng(SEED)
toy_80 = rng.normal(size=(80, 16))
Q_true, _ = np.linalg.qr(rng.normal(size=(16, 16)))
toy_30 = toy_80 @ Q_true
Q_hat, _ = solve_orthogonal_procrustes(toy_80, toy_30)

print("Orthogonality error:", np.linalg.norm(Q_hat.T @ Q_hat - np.eye(16)))
print("Recovery RMSE:", np.sqrt(np.mean((toy_80 @ Q_hat - toy_30) ** 2)))


Orthogonality error: 6.70097204836819e-15
Recovery RMSE: 1.0447476446412983e-15


### Mathematics of Orthogonal Procrustes

The goal is to map the features of a high-altitude matrix ($X_{80}$) into a low-altitude reference space ($X_{30}$) using an orthogonal matrix $R^*$. We solve for the optimal orthogonal mapping matrix $R^*$ that minimizes the sum of squared Euclidean distances (the Frobenius norm) between the mapped and target features
$$R^* = \arg\min_{R^T R = I} \|X_{80} R - X_{30}\|_F^2$$
Where $R^T R = I$ enforces the orthogonality constraint. This constraint is vital because orthogonal transformations preserve vector lengths, angles, and pairwise Euclidean distances.They represent a pure, rigid rotation/ reflection of the feature space without distorting the underlying data structure

**The SVD solution**

According to Schönemann’s generalized solution, this constrained optimization problem has an elegant closed-form solution using Singular Value Decomposition (SVD)
1. First, compute the cross-covariance matrix of your paired calibration sets: $X_{80}^T X_{30}$
2. Calculate its SVD: $$X_{80}^T X_{30} = U \Sigma V^T$$
3. The optimal orthogonal rotation matrix $R^*$ is constructed by multiplying the left and right singular vectors $$R^* = U V^T$$

**Strict vs. Centred Mapping**

two variations of this mathematical transformation:
1. Strict Orthogonal Mapping (No Translation): The high-altitude features are rotated directly using the matrix $$\hat{z}_{30} = z_{80} R^*$$
2. Centred Orthogonal Mapping (With Translation): A common Procrustes workflow centers each domain around its mean vector before rotating, and then translates them into the target space $$\hat{z}_{30} = (z_{80} - \mu_{80}) R^* + \mu_{30}$$

where $\mu_{80}$ and $\mu_{30}$ are the average feature vectors. Comparing these two variants helps establish whether the altitude shift is a pure high-dimensional rotation or if it involves a significant mean shift (translation)


In [6]:
# Production strict and centred Procrustes estimators
@dataclass(frozen=True)
class OrthogonalFeatureMap:
    R: np.ndarray
    source_mean: np.ndarray
    target_mean: np.ndarray
    centred: bool
    singular_values: np.ndarray

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float64)
        return (X - self.source_mean) @ self.R + self.target_mean

    @property
    def orthogonality_error(self) -> float:
        identity = np.eye(self.R.shape[0])
        return float(np.linalg.norm(self.R.T @ self.R - identity, ord="fro"))

def fit_feature_map(
    X_high: np.ndarray,
    X_low: np.ndarray,
    centred: bool = False,
) -> OrthogonalFeatureMap:
    X_high = np.asarray(X_high, dtype=np.float64)
    X_low = np.asarray(X_low, dtype=np.float64)
    if X_high.shape != X_low.shape:
        raise ValueError("Matched high- and low-altitude feature matrices must have equal shape")

    if centred:
        high_mean = X_high.mean(axis=0, keepdims=True)
        low_mean = X_low.mean(axis=0, keepdims=True)
    else:
        high_mean = np.zeros((1, X_high.shape[1]))
        low_mean = np.zeros((1, X_low.shape[1]))

    A, B = X_high - high_mean, X_low - low_mean
    R, singular_values = solve_orthogonal_procrustes(A, B)
    return OrthogonalFeatureMap(R, high_mean, low_mean, centred, singular_values)

def feature_alignment_metrics(X_mapped: np.ndarray, X_target: np.ndarray) -> dict:
    X_mapped = np.asarray(X_mapped, dtype=np.float64)
    X_target = np.asarray(X_target, dtype=np.float64)
    rmse = np.sqrt(np.mean((X_mapped - X_target) ** 2))
    numerator = np.sum(X_mapped * X_target, axis=1)
    denominator = np.linalg.norm(X_mapped, axis=1) * np.linalg.norm(X_target, axis=1)
    cosine = numerator / np.clip(denominator, 1e-12, None)
    return {"rmse": float(rmse), "mean_paired_cosine": float(cosine.mean())}


### Setting Up the Comparison Groups (The 7 Study "Arms")

To rigorously evaluate whether an orthogonal transformation is the correct mathematical model for altitude shifts, this framework structures the feature experiment into seven treatment arms, evaluated on the same held-out physical objects: 
- **Arm A (Source-Domain Reference)**: Fixed 30m linear probe evaluated on real 30m features *Establishes upper-bound reference accuracy*
- **Arm B (Uncorrected Cross-Altitude)**: The 30m probe evaluated directly on raw, uncorrected real 80m features. * Establishes the cross-altitude performance drop*
- **Arm C (Strict Procrustes)**:  Real 80m features mapped by the strict orthogonal matrix ($R^* = U V^T$), then passed to the unchanged 30m probe. *Directly tests the strict orthogonal correction hypothesis*
- **Arm D (Centred Procrustes)**: Real 80m features mapped using the translation inclusive centred method, then passed to the 30m probe. *Tests is a translation (mean shift) is required alongside rotation*
- **Arm E (Mean shift only)**: Adjusting 80m features solely by subtracting $\mu_{80}$ and adding $\mu_{30}$ without any rotation. *Tests if the benefit comes purely from mean alignment*
- **Arm F (Unconstrained Linear Mapping)**: Utilising a regularised linear map (like Ridge regression) trained on the calibration pairs, omitting the orthogonality constraint. *Tests if forcing orthogonality is a useful restriction or if it over-constrains the transformation*
- **Arm G (Negative Control)**: Running the Procrustes calculation after randomly shuffling the pairing identities within the calibrated data. *Proves that the mapping relies on precise, 1-to-1 physical object matching rather than generic domain statistics*


In [7]:
# Seven-arm evaluation. The probe passed here is already trained and remains unchanged.
def classification_metrics(y_true, y_pred) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def evaluate_seven_arms(
    fixed_probe,
    X_low_cal: np.ndarray,
    X_high_cal: np.ndarray,
    X_low_test: np.ndarray,
    X_high_test: np.ndarray,
    y_test: np.ndarray,
    ridge_alpha: float = 1.0,
    seed: int = SEED,
):
    strict_map = fit_feature_map(X_high_cal, X_low_cal, centred=False)
    centred_map = fit_feature_map(X_high_cal, X_low_cal, centred=True)

    mean_shift_test = X_high_test - X_high_cal.mean(0) + X_low_cal.mean(0)

    ridge_map = Ridge(alpha=ridge_alpha, fit_intercept=True)
    ridge_map.fit(X_high_cal, X_low_cal)

    rng = np.random.default_rng(seed)
    shuffled_low = X_low_cal[rng.permutation(len(X_low_cal))]
    shuffled_map = fit_feature_map(X_high_cal, shuffled_low, centred=False)

    arm_features = {
        "A_source_reference": X_low_test,
        "B_uncorrected_high": X_high_test,
        "C_strict_procrustes": strict_map.transform(X_high_test),
        "D_centred_procrustes": centred_map.transform(X_high_test),
        "E_mean_shift_only": mean_shift_test,
        "F_ridge_linear_map": ridge_map.predict(X_high_test),
        "G_shuffled_pair_control": shuffled_map.transform(X_high_test),
    }

    predictions = {
        name: fixed_probe.predict(X_arm) for name, X_arm in arm_features.items()
    }
    results = pd.DataFrame(
        {name: classification_metrics(y_test, pred) for name, pred in predictions.items()}
    ).T
    results.index.name = "arm"

    alignment = pd.DataFrame(
        {
            name: feature_alignment_metrics(X_arm, X_low_test)
            for name, X_arm in arm_features.items()
            if name != "A_source_reference"
        }
    ).T
    alignment.index.name = "arm"

    fitted = {
        "strict": strict_map,
        "centred": centred_map,
        "ridge": ridge_map,
        "shuffled": shuffled_map,
    }
    return results.sort_index(), alignment.sort_index(), predictions, fitted


# Frozen vision encoders

This is the backbone of the Feature-Level correction approach. This approach focusses on how modern, pretrained Vision Foundation Models (VFMs) - such as DINOv3, CLIP, or SAM - internally represent visual scale shifts. In this research, we will use DINOv3 and if compute or model access to DINOv3 is delayed during pilot phase, DINOv2 will be used as a fallback


In [8]:
# Frozen DINOv3 feature extraction through Hugging Face Transformers.
# Loading downloads model weights the first time; later runs use the local cache.
def load_frozen_encoder(model_id: str = MODEL_ID, fallback_id: str | None = FALLBACK_MODEL_ID):
    import torch
    from transformers import AutoImageProcessor, AutoModel

    errors = []
    for candidate in [model_id, fallback_id]:
        if candidate is None:
            continue
        try:
            processor = AutoImageProcessor.from_pretrained(candidate)
            model = AutoModel.from_pretrained(candidate)
            model.eval()
            model.requires_grad_(False)
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            print(f"Loaded {candidate} on {device}")
            return processor, model, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
    raise RuntimeError("Could not load an encoder:\n" + "\n".join(errors))

def extract_global_features(
    image_paths,
    processor,
    model,
    batch_size: int = 16,
) -> np.ndarray:
    import torch

    vectors = []
    paths = [resolve_repo_path(path) for path in image_paths]
    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        images = []
        for path in batch_paths:
            with Image.open(path) as image:
                images.append(image.convert("RGB").copy())
        inputs = processor(images=images, return_tensors="pt")
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.inference_mode():
            outputs = model(**inputs)
        pooled = getattr(outputs, "pooler_output", None)
        if pooled is None:
            hidden = getattr(outputs, "last_hidden_state", None)
            if hidden is None:
                raise AttributeError("Model output has neither pooler_output nor last_hidden_state")
            pooled = hidden[:, 0]  # CLS token for ViT-style encoders
        vectors.append(pooled.detach().float().cpu().numpy())
    features = np.concatenate(vectors, axis=0)
    if not np.isfinite(features).all():
        raise ValueError("Encoder produced NaN or infinite feature values")
    return features

def extract_manifest_features(df: pd.DataFrame, processor, model, batch_size: int = 16):
    image_column = "patch_path" if "patch_path" in df.columns else "image_path"
    features = extract_global_features(df[image_column].tolist(), processor, model, batch_size)
    if len(features) != len(df):
        raise RuntimeError("Feature row count does not match manifest row count")
    return features

def save_feature_bundle(path: Path, df: pd.DataFrame, features: np.ndarray, model_id: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        features=np.asarray(features, dtype=np.float32),
        sample_ids=df["sample_id"].astype(str).to_numpy(),
        object_ids=df["object_id"].astype(str).to_numpy(),
        model_id=np.array(model_id),
        config_json=np.array(json.dumps(CONFIG)),
    )


The encoder must remain frozen for 3 reasons:
1. It isolates the orthogonal correction. If the encoder is trained, better performance might come from fine-tuning rather than from \(R\).
2. It keeps one stable feature space. Procrustes estimates a relationship between 30 m and 80 m features. Changing the encoder also changes both feature spaces.
3. It tests the original practical claim. Question 3 proposes correcting existing features without retraining the vision model.


In [9]:
# Safeguards that prove the encoder is frozen and deterministic for a sample image.
def assert_encoder_frozen(model) -> None:
    if model.training:
        raise AssertionError("Encoder is in training mode; call model.eval()")
    trainable = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    if trainable:
        raise AssertionError(f"Encoder has trainable parameters, first examples: {trainable[:5]}")

def check_feature_repeatability(image_path, processor, model, atol: float = 1e-6) -> float:
    first = extract_global_features([image_path], processor, model, batch_size=1)
    second = extract_global_features([image_path], processor, model, batch_size=1)
    maximum_difference = float(np.max(np.abs(first - second)))
    if maximum_difference > atol:
        raise AssertionError(f"Repeated features differ by {maximum_difference:.3g}")
    return maximum_difference

# After loading the model, run:
# processor, encoder, resolved_model_id = load_frozen_encoder()
# assert_encoder_frozen(encoder)
# check_feature_repeatability("data/patches_fixed_canvas/example.png", processor, encoder)


### How is altitude encoded
Rather than assuming how height affects a model's representation, this research follow the three direct questions to probe the internal structure of these frozen encoders:
1. sparsity (Q1): Is altitude information concentrated in a tiny subset of specific feature channels, or is it spread densely across the entire embedding? *This is measured by analysing activation contrasts across matched low- and high-altitude image pairs*
2. geometry (Q2): does the feature displacement caused by climbing trace an approximately low-rank, axis-aligned direction? *This is measured using Principal Component Analysis (PCA) participation ratios and spectral decay over the altitude sweep*
3. correction (Q3): can a single orthogonal rotation map high-altitude features back into the low-altitude feature space well enough that a simple fixed linear probe can accurately classify them?


In [10]:
# Q1/Q2 diagnostics on exactly matched low/high feature pairs
def altitude_shift_diagnostics(
    X_low: np.ndarray,
    X_high: np.ndarray,
    top_channels: int = 20,
    make_plot: bool = True,
) -> dict:
    X_low = np.asarray(X_low, dtype=np.float64)
    X_high = np.asarray(X_high, dtype=np.float64)
    if X_low.shape != X_high.shape:
        raise ValueError("Low/high feature matrices must contain aligned pairs")

    delta = X_high - X_low
    channel_contrast = np.mean(np.abs(delta), axis=0)
    order = np.argsort(channel_contrast)[::-1]
    total = channel_contrast.sum()
    top_fraction = {
        f"top_{percent}_percent_share": float(
            channel_contrast[order[:max(1, int(np.ceil(len(order) * percent / 100)))]].sum()
            / max(total, 1e-12)
        )
        for percent in (1, 5, 10)
    }

    centred_delta = delta - delta.mean(axis=0, keepdims=True)
    singular_values = np.linalg.svd(centred_delta, compute_uv=False)
    eigenvalues = singular_values ** 2 / max(len(delta) - 1, 1)
    participation_ratio = float(
        eigenvalues.sum() ** 2 / max(np.square(eigenvalues).sum(), 1e-12)
    )
    cumulative_variance = np.cumsum(eigenvalues) / max(eigenvalues.sum(), 1e-12)

    if make_plot:
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        shown = order[:min(top_channels, len(order))]
        axes[0].bar(np.arange(len(shown)), channel_contrast[shown])
        axes[0].set(title="Largest activation contrasts", xlabel="Ranked channel", ylabel="Mean |high-low|")
        axes[1].plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker=".")
        axes[1].axhline(0.90, color="grey", linestyle="--", linewidth=1)
        axes[1].set(title="Altitude-delta spectral decay", xlabel="Principal components", ylabel="Cumulative variance", ylim=(0, 1.02))
        fig.tight_layout()

    return {
        "mean_shift_norm": float(np.linalg.norm(delta.mean(axis=0))),
        "mean_pair_distance": float(np.linalg.norm(delta, axis=1).mean()),
        "participation_ratio": participation_ratio,
        "singular_values": singular_values,
        "channel_contrast": channel_contrast,
        **top_fraction,
    }


### How this connects to study arms

In our previous discussion on Orthogonal Procrustes, we broke down how features are mapped. When evaluating 7 experimental arms, the frozen encoder remains completely untouched.

Linear probe classifier is then trained using the features extracted by the frozen encoder at 30m. When testing at 80m, we will extract the raw features from the frozen encoder, apply the fitted Procrustes rotation matrix directly to those high-dimensional vectors, and pass the corrected vectors to the fixed classifier.

*This directly tests whether altitude transformations are a consistent, transferable orthogonal component within the frozen foundation model's representation space*


In [11]:
# Assemble one feature vector per physical object and altitude.
# Multiple frames are averaged within an object; the object remains the independent unit.
def object_feature_table(df: pd.DataFrame, features: np.ndarray, split: str):
    if len(df) != len(features):
        raise ValueError("Manifest and feature matrix must have the same row count")
    subset = df.loc[df["split"].eq(split)].copy()
    records = []
    for (object_id, altitude_m), rows in subset.groupby(["object_id", "altitude_m"]):
        indices = rows.index.to_numpy()
        labels = rows["class_name"].unique()
        if len(labels) != 1:
            raise ValueError(f"Inconsistent labels for object {object_id}")
        records.append({
            "object_id": str(object_id),
            "altitude_m": float(altitude_m),
            "class_name": labels[0],
            "feature": features[indices].mean(axis=0),
            "n_observations": len(indices),
        })
    return pd.DataFrame(records)

def paired_feature_matrices(
    df: pd.DataFrame,
    features: np.ndarray,
    split: str,
    low_altitude: float = LOW_ALTITUDE_M,
    high_altitude: float = HIGH_ALTITUDE_M,
):
    table = object_feature_table(df, features, split)
    low = table[np.isclose(table["altitude_m"], low_altitude)].set_index("object_id")
    high = table[np.isclose(table["altitude_m"], high_altitude)].set_index("object_id")
    object_ids = low.index.intersection(high.index).sort_values()
    if len(object_ids) == 0:
        raise ValueError(f"No complete low/high pairs found in split '{split}'")
    if not np.array_equal(
        low.loc[object_ids, "class_name"].to_numpy(),
        high.loc[object_ids, "class_name"].to_numpy(),
    ):
        raise ValueError("Class labels disagree within matched pairs")
    return {
        "object_ids": object_ids.to_numpy(),
        "y": low.loc[object_ids, "class_name"].to_numpy(),
        "X_low": np.stack(low.loc[object_ids, "feature"].to_numpy()),
        "X_high": np.stack(high.loc[object_ids, "feature"].to_numpy()),
    }

def source_training_matrix(df: pd.DataFrame, features: np.ndarray):
    table = object_feature_table(df, features, "source_train")
    table = table[np.isclose(table["altitude_m"], LOW_ALTITUDE_M)]
    return np.stack(table["feature"]), table["class_name"].to_numpy(), table["object_id"].to_numpy()


30 m training:

30 m images → Frozen encoder → 30 m features → Train linear probe → Fixed probe

Procrustes calibration:

Matched 30/80 m images → Frozen encoder → Paired features → Fit rotation matrix R

80 m testing:

80 m image → Frozen encoder → Raw feature z₈₀ → Apply R → Corrected feature ẑ₃₀ → Fixed 30 m probe → Prediction


In [12]:
# One compact audit before fitting anything
def audit_q3a_inputs(df: pd.DataFrame, features: np.ndarray) -> pd.DataFrame:
    validate_manifest(df)
    if len(df) != len(features):
        raise ValueError("Feature rows are not aligned with the manifest")
    if features.ndim != 2 or not np.isfinite(features).all():
        raise ValueError("Features must be a finite [samples, dimensions] matrix")

    rows = []
    for split in ("source_train", "calibration", "development", "final_test"):
        subset = df[df["split"].eq(split)]
        altitudes = sorted(map(float, subset["altitude_m"].unique()))
        rows.append({
            "split": split,
            "physical_objects": subset["object_id"].nunique(),
            "samples": len(subset),
            "classes": subset["class_name"].nunique(),
            "altitudes_m": altitudes,
            "feature_dimension": features.shape[1],
        })
    return pd.DataFrame(rows).set_index("split")

# Expected use after feature extraction:
# manifest = pd.read_csv(MANIFEST_PATH)
# audit_q3a_inputs(manifest, features)


## Linear Probing
1. Train at Baseline (30m)
2. Freeze the Probe
3. The uncorrected High-altitude test: pass the uncorrected features captured at 80m through the fixed 30m probe. Because GSD scaling, lens blur and lighting shifts have distorted the 80m feature vectors, the probe's accuracy may drop
4. The corrected High-altitude test: apply the pre-calculated orthogonal procrustes mapping to rotate and translate the 80m features back into the 30m feature space. Then feed these corrected vectors into unchanged 30m probe

### Operational Setup: Oracle-Region Classification
To ensure the linear probe is strictly measuring feature-level distortion and not getting confused by other drone vision errors, the study utilised oracle-region object classification
- A human annotator (or bounding-box ground truth) provides the exact center of a target object (like a parked car, tree, or road marking)
- A fixed-pixel canvas patch is cropped around that center and passed to the frozen encoder to generate a feature vector
- The linear probe then predicts the object's class based on that vector
The strict orthogonal feature correction is considered successful only if the fixed 30m linear probe shows a practically useful improvement in classification accuracy (measured by Macro-F1 or balanced accuracy) on held-out real 80m physical objects after the features are mapped


In [13]:
# Train the 30 m probe once, then estimate paired uncertainty for Arm C versus Arm B.
def fit_fixed_probe(X_train: np.ndarray, y_train: np.ndarray, C: float = 1.0):
    probe = Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=SEED,
        )),
    ])
    return probe.fit(X_train, y_train)

def choose_probe_C(X_train, y_train, X_development, y_development, candidates=(0.01, 0.1, 1.0, 10.0)):
    rows = []
    for C in candidates:
        probe = fit_fixed_probe(X_train, y_train, C=C)
        prediction = probe.predict(X_development)
        rows.append({"C": C, **classification_metrics(y_development, prediction)})
    results = pd.DataFrame(rows).sort_values(["balanced_accuracy", "macro_f1"], ascending=False)
    return float(results.iloc[0]["C"]), results

def paired_object_bootstrap(
    y_true,
    baseline_pred,
    corrected_pred,
    object_ids,
    metric="balanced_accuracy",
    repetitions: int = 5000,
    seed: int = SEED,
) -> dict:
    y_true = np.asarray(y_true)
    baseline_pred = np.asarray(baseline_pred)
    corrected_pred = np.asarray(corrected_pred)
    object_ids = np.asarray(object_ids)
    labels = np.unique(y_true)

    def score(y, pred):
        if metric == "balanced_accuracy":
            return balanced_accuracy_score(y, pred)
        if metric == "macro_f1":
            return f1_score(y, pred, labels=labels, average="macro", zero_division=0)
        raise ValueError("metric must be 'balanced_accuracy' or 'macro_f1'")

    unique_objects = np.unique(object_ids)
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(repetitions):
        sampled_objects = rng.choice(unique_objects, size=len(unique_objects), replace=True)
        sampled_indices = np.concatenate([np.flatnonzero(object_ids == obj) for obj in sampled_objects])
        differences.append(
            score(y_true[sampled_indices], corrected_pred[sampled_indices])
            - score(y_true[sampled_indices], baseline_pred[sampled_indices])
        )
    observed = score(y_true, corrected_pred) - score(y_true, baseline_pred)
    low, high = np.quantile(differences, [0.025, 0.975])
    return {
        "metric": metric,
        "observed_delta_C_minus_B": float(observed),
        "ci_95_low": float(low),
        "ci_95_high": float(high),
        "bootstrap_unit": "physical object",
    }

def run_q3a(manifest: pd.DataFrame, features: np.ndarray, ridge_alpha: float = 1.0):
    X_train, y_train, _ = source_training_matrix(manifest, features)
    development = paired_feature_matrices(manifest, features, "development")
    selected_C, tuning = choose_probe_C(
        X_train, y_train, development["X_low"], development["y"]
    )
    fixed_probe = fit_fixed_probe(X_train, y_train, C=selected_C)

    calibration = paired_feature_matrices(manifest, features, "calibration")
    final_test = paired_feature_matrices(manifest, features, "final_test")
    results, alignment, predictions, fitted = evaluate_seven_arms(
        fixed_probe,
        calibration["X_low"], calibration["X_high"],
        final_test["X_low"], final_test["X_high"], final_test["y"],
        ridge_alpha=ridge_alpha,
    )
    uncertainty = paired_object_bootstrap(
        final_test["y"],
        predictions["B_uncorrected_high"],
        predictions["C_strict_procrustes"],
        final_test["object_ids"],
    )
    return {
        "probe": fixed_probe,
        "selected_C": selected_C,
        "probe_tuning": tuning,
        "classification_results": results,
        "alignment_results": alignment,
        "predictions": predictions,
        "fitted_maps": fitted,
        "primary_uncertainty": uncertainty,
    }

# Final use:
# experiment = run_q3a(manifest, features)
# display(experiment["classification_results"])
# display(experiment["alignment_results"])
# experiment["primary_uncertainty"]


## Ground sampling distance
### The linear scaling rule
GSD scales linearly with flight altitude:
- The 30m to 80m Jump: Because an 80m altitude is approximately 2.66 times higher than 30m, spatial resolution drops by that exact linear factor
- Target Object Scale Ratio: This yields an expected linear object-scale ratio of $s = 30/80 = 0.375$ and an area scale ratio of $s^2 = 0.140625$. Consequently, an object’s pixel width and height at 80m are only 37.5% of their 30m values, and its physical pixel area shrinks to just 14.1% before accounting for other atmospheric or lens effects

### Navigating the DJI Tello Hardware Roadblock
DJI Tello drone introduces immediate hardware constraints:
- Published Limit: The Tello specifications list a maximum flight height of 30m
- Fixed Optics:  The Tello features a fixed camera with no mechanical gimbal adjustment
- No GPS: It relies entirely on optical flow and visual positioning, making GPS logging impossible

### Methodological Pivot (10m to 25m adaptation)
adapt flight heights into a 10m to 25m altitude jump:
1. Linear Scaling Factor: The scaling ratio for this pivot is exactly $2.5$ ($25/10 = 2.5$). At 25m, each pixel covers 2.5 times more ground distance in each linear direction and 6.25 times more ground area than at 10m
2. Tello GSD Calculations: Utilizing the Tello's camera specifications (5-Megapixel sensor, 2592 pixels image width, and 82.6° FOV):
- At 10m: $\text{Ground Width} \approx 17.57\text{m} \implies \text{GSD} \approx \mathbf{0.68\text{ cm/pixel}}$
- At 25m: $\text{Ground Width} \approx 43.93\text{m} \implies \text{GSD} \approx \mathbf{1.69\text{ cm/pixel}}$

### Geometric & Field-of-View (FOV) Limitations
- The Planar Assumption: The GSD calculation assumes a completely flat terrain and a nadir (90° straight down) camera view. It does not naturally account for ground topography, object height, oblique camera angles, or dynamic changes in sensor settings
- The Field-of-View Mismatch: As a drone ascends, the camera sees significantly more ground. At 80m, the camera captures 2.667 times more ground linearly and 7.11 times more ground area than at 30m. Simply shrinking a 30m frame cannot recreate the surrounding spatial context that was never captured. -> fixed-size native-pixel tiles extracted from a broader 30m image coverage map


In [14]:
# Camera footprint and approximate GSD from horizontal field of view.
# These are pinhole/flat-ground/nadir calculations and must be checked in real images.
def ground_footprint_width_m(height_m: float, horizontal_fov_deg: float) -> float:
    return 2.0 * height_m * np.tan(np.deg2rad(horizontal_fov_deg) / 2.0)

def approximate_gsd_cm_per_pixel(
    height_m: float,
    image_width_px: int,
    horizontal_fov_deg: float,
) -> float:
    width_m = ground_footprint_width_m(height_m, horizontal_fov_deg)
    return width_m / image_width_px * 100.0

def altitude_scale_summary(low_m: float, high_m: float) -> dict:
    linear_ratio = low_m / high_m
    return {
        "high_to_low_object_width_ratio": linear_ratio,
        "high_to_low_object_area_ratio": linear_ratio ** 2,
        "high_vs_low_gsd_multiplier": high_m / low_m,
        "high_vs_low_pixel_ground_area_multiplier": (high_m / low_m) ** 2,
    }

TELLO = {
    "max_flight_height_m": 30.0,
    "photo_width_px": 2592,
    "horizontal_fov_deg": 82.6,
    "vision_positioning_best_below_m": 6.0,
}

rows = []
for altitude_m in (LOW_ALTITUDE_M, HIGH_ALTITUDE_M):
    rows.append({
        "altitude_m": altitude_m,
        "ground_width_m": ground_footprint_width_m(altitude_m, TELLO["horizontal_fov_deg"]),
        "approx_gsd_cm_per_pixel": approximate_gsd_cm_per_pixel(
            altitude_m, TELLO["photo_width_px"], TELLO["horizontal_fov_deg"]
        ),
    })
gsd_table = pd.DataFrame(rows)
display(gsd_table.round(3))
print(altitude_scale_summary(LOW_ALTITUDE_M, HIGH_ALTITUDE_M))

if HIGH_ALTITUDE_M > TELLO["max_flight_height_m"]:
    raise ValueError("Configured high altitude exceeds the published Tello maximum")
if HIGH_ALTITUDE_M > TELLO["vision_positioning_best_below_m"]:
    print("Flight-design note: this height is above the range where the Tello manual says vision positioning works best.")


,altitude_m,ground_width_m,approx_gsd_cm_per_pixel
0,10.0,17.570,0.678
1,25.0,43.926,1.695


{'high_to_low_object_width_ratio': 0.4, 'high_to_low_object_area_ratio': 0.16000000000000003, 'high_vs_low_gsd_multiplier': 2.5, 'high_vs_low_pixel_ground_area_multiplier': 6.25}
Flight-design note: this height is above the range where the Tello manual says vision positioning works best.
